In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots

include("functions.jl")
Random.seed!(2025)
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000

Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]

tau = length(Istar_obs)

model_tag_sym = :sliding
KMAX_UPPER = 30  
KMAX_fixed = 26

# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 1

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER,
            k_max_fixed = KMAX_fixed)
    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [sliding_model] Fitting chain 1 (tau=34)
[ Info: [sliding] iter 1000/1000000 elapsed=3.7s, rate=0.079, mean=[1.240, 0.00027, 1.496], std=[0.1685, 0.000287, 0.0089] [ADAPT]
[ Info: [sliding] iter 2000/1000000 elapsed=6.8s, rate=0.045, mean=[1.658, 0.00018, 1.475], std=[0.3968, 0.000226, 0.0204] [ADAPT]
[ Info: [sliding] iter 3000/1000000 elapsed=9.0s, rate=0.033, mean=[1.841, 0.00015, 1.466], std=[0.3985, 0.000196, 0.0202] [ADAPT]
[ Info: [sliding] iter 4000/1000000 elapsed=11.2s, rate=0.028, mean=[1.968, 0.00013, 1.459], std=[0.3976, 0.000177, 0.0206] [ADAPT]
[ Info: [sliding] iter 5000/1000000 elapsed=13.4s, rate=0.024, mean=[2.029, 0.00012, 1.457], std=[0.3745, 0.000165, 0.0195] [ADAPT]
[ Info: [sliding] iter 6000/1000000 elapsed=15.6s, rate=0.023, mean=[2.060, 0.00012, 1.457], std=[0.3481, 0.000155, 0.0182] [ADAPT]
[ Info: [sliding] iter 7000/1000000 elapsed=17.8s, rate=0.022, mean=[2.100, 0.00011, 1.462], std=[0.3342, 0.000148, 0.0197] [ADAPT]
[ Info: [sliding] iter 8000/10